Lab 6: Random Forest
Jorge Angon, Faith Yang

https://www.kaggle.com/datasets/dhrubangtalukdar/cybersecurity-threat-detection-dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics

df= pd.read_csv('cybersecurity.csv')
df.head()

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type
0,2025-10-01 00:12:54,188.176.27.165,253.240.113.218,56377,445,TCP,8029,17204,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/login?id=385071,False,0,benign
1,2025-10-01 00:23:43,68.59.26.43,212.75.38.111,51165,1433,TCP,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,False,0,benign
2,2025-10-01 00:25:46,119.204.243.78,90.28.90.234,14948,1433,TCP,316502,38571,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,False,0,benign
3,2025-10-01 00:27:21,122.119.194.175,175.140.78.230,36097,443,TCP,70933,21935,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=114701,False,0,benign
4,2025-10-01 00:40:09,181.199.242.68,55.99.177.69,445,21255,TCP,12721,9939,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/config.php?id=345569,False,0,benign


In [ ]:
# Make text columns usable for the model
df["protocol_code"] = df["protocol"].map({"TCP": 0, "UDP": 1, "ICMP": 2})
df["is_internal_traffic_code"] = df["is_internal_traffic"].map({"False": 0, "True": 1, False: 0, True: 1})

# Split dataset in features and target variable
x_data = df[["src_port", "dst_port", "protocol_code", "bytes_sent", "bytes_received", "is_internal_traffic_code"]]
y_data = df["label"]

In [ ]:
# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=1)

In [ ]:
# Train a Random Forest with random feature selection
rf_model = RandomForestClassifier(n_estimators=5, max_features=2, random_state=42)

rf_model.fit(X_train, y_train)

# Calculate the accuracy and assign it to rf_accuracy
rf_accuracy = rf_model.score(X_test, y_test)

print("RF Model Accuracy:", f"{rf_accuracy * 100:.2f}", "%")

RF Model Accuracy: 96.43 %


In [ ]:
# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)

# Calculate precision, recall, and F1-score for Random Forest
precision_rf = metrics.precision_score(y_test, y_pred_rf)
recall_rf = metrics.recall_score(y_test, y_pred_rf)
f1_rf = metrics.f1_score(y_test, y_pred_rf)

print("RF Precision:", precision_rf)
print("RF Recall:", recall_rf)
print("RF F1-Score:", f1_rf)

RF Precision: 0.6140350877192983
RF Recall: 0.2916666666666667
RF F1-Score: 0.3954802259887006


In [ ]:
# Prepare a provision for a new data entry and its prediction of an outcome.

def predict_activity():
    while True:
        try:
            src_port = int(input("Enter source port: "))
            dst_port = int(input("Enter destination port: "))
            protocol_code = int(input("Enter protocol code (TCP=0, UDP=1, ICMP=2): "))
            bytes_sent = int(input("Enter bytes sent: "))
            bytes_received = int(input("Enter bytes received: "))
            is_internal_traffic_code = int(input("Enter internal traffic? (No=0, Yes=1): "))

            # Input validation
            if any(val < 0 for val in [src_port, dst_port, protocol_code, bytes_sent, bytes_received, is_internal_traffic_code]):
                print("Error: Please enter non-negative values for all features.")
                continue

            new_activity_data = pd.DataFrame([[src_port, dst_port, protocol_code, bytes_sent, bytes_received, is_internal_traffic_code]],
                                             columns=["src_port", "dst_port", "protocol_code", "bytes_sent", "bytes_received", "is_internal_traffic_code"])

            prediction = rf_model.predict(new_activity_data)[0]

            if prediction == 1:
                print("Prediction: Abnormal / Attack Activity")
            else:
                print("Prediction: Normal Activity")

            break

        except ValueError:
            print("Error: Invalid input. Please enter numeric values only.")

predict_activity()

Enter source port: 56377
Enter destination port: 443
Enter protocol code (TCP=0, UDP=1, ICMP=2): 0
Enter bytes sent: 8029
Enter bytes received: 9939
Enter internal traffic? (No=0, Yes=1): 1
Prediction: Normal Activity


In [ ]:
# Show a table of features and its corresponding values

feature_importances = pd.DataFrame({"feature": x_data.columns, "importance": rf_model.feature_importances_})
feature_importances = feature_importances.sort_values("importance", ascending=False)
feature_importances

,feature,importance
3,bytes_sent,0.279958
4,bytes_received,0.250891
1,dst_port,0.224612
0,src_port,0.210097
2,protocol_code,0.017710
5,is_internal_traffic_code,0.016732


For this dataset, we used a random forest model to predict whether cybersecurity activity was normal or abnormal. The model uses the source port, destination port, protocol, bytes sentm bytes recieved and whether the traffic was internal or not. Based on these results, the model has an accuracy of 96.34% which means that it was correct most the time. The most important feature were bytes sent, bytes recieved, destiantion port, and source port. The remaning features had less influence on prediction. Even though the overall accuracy was strong, the recall score was lower which means that the model was better at identifying normal activity than catching every abnormal case.